# 01 — Data Preparation

Load BIRD/Spider benchmarks, filter SynSQL, produce stratified SFT subsets.

In [ ]:
import os, sys
from pathlib import Path

# Adjust if running on Kaggle vs Colab
REPO_DIR = Path.cwd()
if 'SLM-PRJ' not in str(REPO_DIR):
    REPO_DIR = Path('/kaggle/working/SLM-PRJ')
os.chdir(REPO_DIR)
sys.path.insert(0, str(REPO_DIR))

DATA_DIR = REPO_DIR.parent / 'data'
BIRD_DIR = DATA_DIR / 'bird'
SPIDER_DIR = DATA_DIR / 'spider'
SYNSQL_RAW = DATA_DIR / 'synsql' / 'raw.jsonl'
OUTPUT_DIR = DATA_DIR / 'sft_subsets'

print(f'Repo: {REPO_DIR}')
print(f'Data: {DATA_DIR}')

## 1. Load BIRD dev

In [ ]:
from benchmarks.bird_loader import load_bird_dev, get_db_path as bird_db_path
from collections import Counter

bird_dev = load_bird_dev(BIRD_DIR)
print(f'BIRD dev: {len(bird_dev)} questions')
print(f'Databases: {len(set(r["db_id"] for r in bird_dev))}')
print('Difficulty distribution:')
for k, v in sorted(Counter(r['difficulty'] for r in bird_dev).items()):
    print(f'  {k}: {v}')

## 2. Load Spider dev

In [ ]:
from benchmarks.spider_loader import load_spider_dev, get_db_path as spider_db_path

spider_dev = load_spider_dev(SPIDER_DIR)
print(f'Spider dev: {len(spider_dev)} questions')
print(f'Databases: {len(set(r["db_id"] for r in spider_dev))}')

## 3. Sanity check — EX gold vs gold

In [ ]:
from benchmarks.eval_ex import execution_match
import time

# Test on first 50 BIRD dev questions
sample = bird_dev[:50]
start = time.time()
results = []
for r in sample:
    db = bird_db_path(BIRD_DIR, r['db_id'])
    matched = execution_match(r['gold_sql'], r['gold_sql'], db)
    results.append(matched)

elapsed = time.time() - start
acc = sum(results) / len(results)
print(f'Gold vs gold (first 50): {acc:.2%} ({sum(results)}/{len(results)})')
print(f'Time: {elapsed:.1f}s ({elapsed/len(results):.2f}s/query)')
assert acc == 1.0, f'Gold vs gold should be 100%, got {acc:.2%}'

## 4. Filter SynSQL and produce SFT subsets

In [ ]:
import logging, json
logging.basicConfig(level=logging.INFO)

from training.data_prep import prepare_subsets

stats = prepare_subsets(
    raw_path=SYNSQL_RAW,
    output_dir=OUTPUT_DIR,
    sizes=(10_000, 30_000, 50_000),
    max_chars=8192,
    seed=42,
)
print(json.dumps(stats, indent=2))

## 5. Verify prompt formatting

In [ ]:
from training.prompt_template import format_sft_messages
from benchmarks.bird_loader import get_schema

example = bird_dev[0]
schema = get_schema(BIRD_DIR, example['db_id'])
messages = format_sft_messages(
    question=example['question'],
    schema=schema,
    gold_sql=example['gold_sql'],
    reasoning='Let me think step by step...',
    evidence=example.get('evidence', ''),
)

for msg in messages:
    print(f'--- {msg["role"]} ---')
    print(msg['content'][:300])
    print()